# 01 — Feature Engineering
## Stationarity, Differencing, Lags & Rolling Statistics

### Purpose
Test stationarity of all indicators using ADF and KPSS tests.
First-difference non-stationary columns. Build lag features (t-1).
Compute rolling mean and standard deviation (3yr, 5yr).

### Input
- `../data/01_panel_cleaned.csv`

### Output
- `../data/02_panel_features.csv`

### Run after → Run before
`00_data_preparation.ipynb` → `02_instability_index_eda.ipynb`

In [70]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

df_panel = pd.read_csv("data/01_panel_cleaned.csv")
print(f"Loaded: {df_panel.shape} | Countries: {df_panel['COUNTRY'].nunique()}")
df_panel.head()

Loaded: (5052, 23) | Countries: 177


,COUNTRY,YEAR,Inflation,Current_Account,Expenditure,Exports,Investment,Debt,GDP_Growth,Savings,...,Inflation_imputed,Current_Account_imputed,Expenditure_imputed,Exports_imputed,Imports_imputed,Savings_imputed,Investment_imputed,Debt_imputed,Fiscal_Balance_imputed,Revenue_imputed
0,"Afghanistan, Islamic Republic of",2003,35.663,29.616,11.927,49.541,30.102,270.602,8.692,59.718,...,False,False,False,False,False,False,False,False,False,False
1,"Afghanistan, Islamic Republic of",2004,16.358,37.216,15.069,-8.436,35.354,244.967,0.671,72.570,...,False,False,False,False,False,False,False,False,False,False
2,"Afghanistan, Islamic Republic of",2005,10.569,30.226,15.651,41.968,37.048,206.356,11.830,67.274,...,False,False,False,False,False,False,False,False,False,False
3,"Afghanistan, Islamic Republic of",2006,6.785,20.844,18.262,-6.919,29.489,22.985,5.361,50.333,...,False,False,False,False,False,False,False,False,False,False
4,"Afghanistan, Islamic Republic of",2007,8.681,63.390,21.448,-11.904,55.852,20.137,13.340,119.243,...,False,False,False,False,False,False,False,False,False,False


In [71]:
# ── Stationarity tests (ADF + KPSS) ──────────────────────────
from statsmodels.tsa.stattools import adfuller, kpss
import warnings
warnings.filterwarnings("ignore")

cols_to_test = [
    "GDP_Growth","Inflation","Debt","Fiscal_Balance",
    "Current_Account","Exports","Imports",
    "Revenue","Expenditure","Savings","Investment"
]

results = []
for col in cols_to_test:
    adf_stats, kpss_stats = [], []
    for country, grp in df_panel.groupby("COUNTRY"):
        series = grp[col].dropna()
        if len(series) < 8:
            continue
        try:
            adf_stats.append(adfuller(series, autolag="AIC")[1])
        except: pass
        try:
            kpss_stats.append(kpss(series, regression="c", nlags="auto")[1])
        except: pass
    results.append({
        "Column"         : col,
        "ADF_pct_reject" : np.mean(np.array(adf_stats) < 0.05) * 100,
        "KPSS_pct_reject": np.mean(np.array(kpss_stats) < 0.05) * 100,
        "ADF_median_p"   : np.median(adf_stats),
        "KPSS_median_p"  : np.median(kpss_stats)
    })

stationarity_df = pd.DataFrame(results)
print(stationarity_df.to_string(index=False))

         Column  ADF_pct_reject  KPSS_pct_reject  ADF_median_p  KPSS_median_p
     GDP_Growth       76.271186        15.254237      0.001378       0.100000
      Inflation       61.581921        19.209040      0.019170       0.100000
           Debt       17.514124        46.327684      0.414416       0.075151
 Fiscal_Balance       42.937853        22.598870      0.074857       0.100000
Current_Account       31.638418        22.033898      0.167608       0.100000
        Exports       83.615819        11.864407      0.000245       0.100000
        Imports       85.875706         7.909605      0.000162       0.100000
        Revenue       27.683616        42.937853      0.254928       0.079732
    Expenditure       25.423729        45.762712      0.230529       0.076222
        Savings       24.858757        40.112994      0.205685       0.076535
     Investment       32.768362        40.112994      0.181276       0.100000


In [72]:
# ── First-difference non-stationary columns ───────────────────
non_stationary_cols = ["Debt","Expenditure","Revenue","Savings","Investment"]

for col in non_stationary_cols:
    df_panel[f"{col}_diff"] = df_panel.groupby("COUNTRY")[col]        .transform(lambda x: x.diff())

# Verify stationarity after differencing
for col in non_stationary_cols:
    pvals = []
    for country, grp in df_panel.groupby("COUNTRY"):
        s = grp[f"{col}_diff"].dropna()
        if len(s) >= 8:
            try: pvals.append(adfuller(s, autolag="AIC")[1])
            except: pass
    pct = np.mean(np.array(pvals) < 0.05) * 100
    print(f"{col}_diff → {pct:.1f}% stationary after differencing")

Debt_diff → 70.6% stationary after differencing
Expenditure_diff → 85.3% stationary after differencing
Revenue_diff → 82.5% stationary after differencing
Savings_diff → 80.8% stationary after differencing
Investment_diff → 85.3% stationary after differencing


In [73]:
# ── Rolling statistics (3yr and 5yr) ─────────────────────────
roll_cols = [
    "Inflation","GDP_Growth","Exports","Imports",
    "Fiscal_Balance","Current_Account",
    "Debt","Expenditure","Revenue","Savings","Investment"
]

for col in roll_cols:
    for w in [3, 5]:
        df_panel[f"{col}_rollmean{w}"] = df_panel.groupby("COUNTRY")[col]            .transform(lambda x: x.shift(1).rolling(w, min_periods=2).mean())
        df_panel[f"{col}_rollstd{w}"] = df_panel.groupby("COUNTRY")[col]            .transform(lambda x: x.shift(1).rolling(w, min_periods=2).std())

print(f"Rolling stats added. Total columns: {df_panel.shape[1]}")

Rolling stats added. Total columns: 72


In [74]:
stationary_cols  = ["GDP_Growth","Inflation","Exports","Imports",
                    "Fiscal_Balance","Current_Account"]
differenced_cols = ["Debt_diff","Expenditure_diff","Revenue_diff",
                    "Savings_diff","Investment_diff"]
all_feature_cols = stationary_cols + differenced_cols

for col in all_feature_cols:
    df_panel[f"{col}_lag1"] = df_panel.groupby("COUNTRY")[col]\
        .transform(lambda x: x.shift(1))

# Autoregressive features
df_panel["GDP_Growth_lag1"]      = df_panel.groupby("COUNTRY")["GDP_Growth"]\
    .transform(lambda x: x.shift(1))
df_panel["GDP_Growth_rollmean3"] = df_panel.groupby("COUNTRY")["GDP_Growth"]\
    .transform(lambda x: x.shift(1).rolling(3, min_periods=2).mean())
df_panel["Inflation_lag1_log"]   = np.sign(df_panel["Inflation_lag1"]) * \
    np.log1p(np.abs(df_panel["Inflation_lag1"]))

lag_feature_cols = [f"{col}_lag1" for col in all_feature_cols]
print(f"Lag features created: {len(lag_feature_cols)}")
print(f"Total columns: {df_panel.shape[1]}")

Lag features created: 11
Total columns: 84


In [75]:
# 1. Create the list (which currently contains duplicates)
raw_new_cols = lag_feature_cols + \
    [c for c in df_panel.columns if "rollmean" in c or "rollstd" in c] + \
    ["GDP_Growth_lag1", "GDP_Growth_rollmean3", "Inflation_lag1_log"]

# 2. REMOVE DUPLICATES (This prevents the ValueError)
new_cols = list(set(raw_new_cols))

# 3. Year-wise median → country ffill/bfill → global median
df_panel[new_cols] = df_panel.groupby("YEAR")[new_cols]\
    .transform(lambda x: x.fillna(x.median(numeric_only=True)))

df_panel[new_cols] = df_panel.groupby("COUNTRY")[new_cols]\
    .transform(lambda x: x.ffill().bfill())

for col in new_cols:
    df_panel[col] = df_panel[col].fillna(df_panel[col].median())

# 4. Drop only where target is missing
df_panel = df_panel.dropna(subset=["GDP_Growth"])

print(f"Final shape  : {df_panel.shape}")
print(f"Countries    : {df_panel['COUNTRY'].nunique()}")
print(f"Feature NaN  : {df_panel[new_cols].isnull().sum().sum()}")

Final shape  : (5052, 84)
Countries    : 177
Feature NaN  : 0


In [76]:
# ── Save checkpoint ───────────────────────────────────────────
df_panel.to_csv("data/02_panel_features.csv", index=False)
print("Saved: data/02_panel_features.csv")
print(df_panel.shape)

Saved: data/02_panel_features.csv
(5052, 84)
